# Automated Research Pipeline — Multi-Stage Analysis with Self-Correction

## Overview

This notebook implements a **research synthesis pipeline** that mirrors how a skilled analyst would approach a complex research question: gather sources, summarise them, fact-check the claims, critique the quality, revise until satisfied, and produce a polished report.

### Research Topic

> **"Impact of Large Language Models on Software Development Productivity"**

### What you will learn

| Concept | Where demonstrated |
|---------|--------------------|
| `GraphRunner` + `GraphNodeSpec` | DAG-based research workflow |
| `EpisodicMemory` | Accumulating context across iterations |
| `MapReduceCoordinator` | Parallel summarise + fact-check |
| `AssemblyCoordinator` | Sequenced quality critique + report |
| MCMC refinement loop | Draft → critique → revise until confidence ≥ threshold |
| Ensemble agreement score | Measuring research confidence |
| `StructuredLogger` | Observability throughout |

### Pipeline Architecture

```
TopicExtractor
      │
      ▼
SourceSearcher (mock)
      │
   ┌──┴──┐  (parallel)
   ▼     ▼
Summarize  FactCheck
   └──┬──┘
      │ AggregatorAgent
      ▼
QualityCritic ◄──────┐
      │               │ (revise loop)
      ▼               │
ReportGenerator ──────┘ (if quality < threshold)
```

In [ ]:
import json
import random
import statistics
import copy
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

from agentic_codex import (
    AgentBuilder,
    Context,
    FunctionAdapter,
    EpisodicMemory,
    MemoryCapability,
)
from agentic_codex.core.schemas import AgentStep, Message
from agentic_codex.patterns import (
    MapReduceCoordinator,
    AssemblyCoordinator, Stage,
    GraphRunner, GraphNodeSpec,
)
from agentic_codex.core.observability.logger import StructuredLogger

random.seed(7)
logger = StructuredLogger(name='research_pipeline')
print('Imports OK.')

## Section 1 — Mock Research Data

We simulate a corpus of 8 research sources with abstracts, author credibility scores, and publication year. In production `SourceSearcher` would call Semantic Scholar, arXiv, or an internal knowledge base.

In [ ]:
RESEARCH_TOPIC = "Impact of Large Language Models on Software Development Productivity"

MOCK_SOURCES = [
    {
        'id': 'src_001',
        'title': 'GitHub Copilot Study: 55% Faster Task Completion',
        'authors': ['Dohmke, T.', 'Fischer, M.', 'Schafer, B.'],
        'year': 2022,
        'venue': 'GitHub Research Report',
        'credibility_score': 0.72,
        'abstract': 'Controlled experiment with 95 developers; Copilot users completed task 55.8% faster on average. Tasks were self-contained functions with clear docstrings.',
        'key_finding': 'LLM-assisted coding increases speed 55% on isolated tasks',
        'methodology': 'RCT',
        'sample_size': 95,
    },
    {
        'id': 'src_002',
        'title': 'Productivity Gains from AI Pair Programming: A Meta-Analysis',
        'authors': ['Zhang, Q.', 'Liu, J.'],
        'year': 2023,
        'venue': 'ACM TOSEM',
        'credibility_score': 0.89,
        'abstract': 'Meta-analysis of 14 studies (n=1,240 developers). Average productivity improvement 23-48% depending on task complexity. Complex architectural tasks show minimal gains.',
        'key_finding': '23-48% productivity gain on routine coding; complex design work sees <5% improvement',
        'methodology': 'meta-analysis',
        'sample_size': 1240,
    },
    {
        'id': 'src_003',
        'title': 'The Hidden Costs: LLM Code Quality and Maintenance Debt',
        'authors': ['Torres, R.', 'Singh, A.'],
        'year': 2023,
        'venue': 'ICSE 2023',
        'credibility_score': 0.85,
        'abstract': 'Longitudinal study tracking 600 AI-generated code snippets over 6 months. 23% introduced subtle bugs; maintenance burden increased 18% in teams over-relying on LLM output.',
        'key_finding': 'LLM code has 23% higher bug rate; teams need robust review processes',
        'methodology': 'longitudinal_study',
        'sample_size': 600,
    },
    {
        'id': 'src_004',
        'title': 'Junior vs Senior Developer Gains from AI Assistance',
        'authors': ['Peng, S.', 'Kalliamvakou, E.', 'Cihon, P.'],
        'year': 2023,
        'venue': 'MIT Sloan Working Paper',
        'credibility_score': 0.81,
        'abstract': 'Junior developers (<2 years) show 35-50% speed gains; senior developers (>8 years) show 8-15% gains on familiar problem types. LLMs reduce experience gap.',
        'key_finding': 'LLMs disproportionately benefit junior developers; reduce experience gap',
        'methodology': 'observational_study',
        'sample_size': 243,
    },
    {
        'id': 'src_005',
        'title': 'Developer Satisfaction and Over-Reliance on AI Tools',
        'authors': ['Liang, J.', 'Yang, Y.'],
        'year': 2024,
        'venue': 'CHI 2024',
        'credibility_score': 0.78,
        'abstract': 'Survey of 1,200 developers. 67% report increased job satisfaction; 31% express concern about skill atrophy; 44% admit accepting LLM suggestions without understanding them.',
        'key_finding': 'Satisfaction up 67%, but skill atrophy is a growing concern among 31% of developers',
        'methodology': 'survey',
        'sample_size': 1200,
    },
]

print(f'Loaded {len(MOCK_SOURCES)} research sources on: "{RESEARCH_TOPIC}"')
for s in MOCK_SOURCES:
    print(f"  [{s['id']}] {s['title'][:55]}... ({s['year']}) cred={s['credibility_score']:.2f}")

## Section 2 — Agent Definitions

### 2.1 TopicExtractor — Decompose research question into sub-topics

In [ ]:
def _topic_extractor_step(ctx: Context) -> AgentStep:
    """
    Decompose a research question into structured sub-topics.
    
    REAL LLM VERSION: sends topic to GPT-4o with system prompt:
    "You are a research analyst. Decompose this topic into 4-6 measurable
    sub-questions that can be answered from academic literature."
    """
    topic = ctx.scratch.get('research_topic', ctx.goal)
    
    # Mock: static decomposition
    sub_topics = [
        'Productivity measurement: speed, output volume, defect rate',
        'Code quality: correctness, maintainability, test coverage',
        'Developer experience: satisfaction, cognitive load, skill development',
        'Team dynamics: junior/senior gap, collaboration patterns',
        'Long-term effects: skill atrophy, job market implications',
    ]
    
    result = {
        'original_topic': topic,
        'sub_topics': sub_topics,
        'search_queries': [
            f'"LLM" OR "AI assistant" {t.split(":")[0].lower()}'
            for t in sub_topics
        ],
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'topic_analysis': result},
        stop=False,
    )

topic_extractor = AgentBuilder('TopicExtractor', 'research-analyst').with_step(_topic_extractor_step).build()
print('TopicExtractor built.')

In [ ]:
# 2.2 SourceSearcher — Returns mock corpus

def _source_searcher_step(ctx: Context) -> AgentStep:
    """
    Search for relevant sources.
    
    REAL VERSION: calls Semantic Scholar API / arXiv / internal vector DB.
    Ranks results by relevance + credibility score.
    """
    queries = ctx.scratch.get('topic_analysis', {}).get('search_queries', [])
    n_queries = len(queries)
    
    # Return our mock corpus, simulating retrieval
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps({'sources': MOCK_SOURCES}))],
        state_updates={'retrieved_sources': MOCK_SOURCES, 'query_count': n_queries},
        stop=False,
    )

source_searcher = AgentBuilder('SourceSearcher', 'retrieval-agent').with_step(_source_searcher_step).build()

# 2.3 SummarySynthesizer — Generates coherent synthesis from sources

def _summarizer_step(ctx: Context) -> AgentStep:
    """
    Synthesise sources into a coherent narrative summary.
    
    REAL LLM VERSION: passes all source abstracts + key findings to GPT-4o
    with a prompt to write a 500-word structured synthesis.
    """
    sources = ctx.scratch.get('retrieved_sources', MOCK_SOURCES)
    
    # Mock synthesis
    key_findings = [s['key_finding'] for s in sources]
    avg_credibility = sum(s['credibility_score'] for s in sources) / len(sources)
    total_n = sum(s.get('sample_size', 0) for s in sources)
    
    synthesis = {
        'agent': 'SummarySynthesizer',
        'summary': (
            'The evidence strongly suggests LLMs improve raw coding speed (23-55%) but the benefits '
            'are unevenly distributed. Junior developers gain the most; complex architectural work '
            'shows minimal improvement. Code quality and maintenance debt are legitimate concerns '
            'requiring careful process design around AI-generated code.'
        ),
        'key_findings': key_findings,
        'sources_used': len(sources),
        'total_sample_size': total_n,
        'avg_source_credibility': round(avg_credibility, 3),
        'synthesis_confidence': round(avg_credibility * 0.9, 3),
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(synthesis))],
        state_updates={'synthesis': synthesis},
        stop=False,
    )

summarizer = AgentBuilder('SummarySynthesizer', 'synthesiser').with_step(_summarizer_step).build()
print('SourceSearcher + SummarySynthesizer built.')

In [ ]:
# 2.4 FactChecker — Validates claims against source data

def _fact_checker_step(ctx: Context) -> AgentStep:
    """
    Cross-checks claims in the synthesis against source data.
    
    REAL LLM VERSION: uses GPT-4o with each source as context to verify
    statistical claims, flag unsupported assertions, and check citation accuracy.
    """
    sources = ctx.scratch.get('retrieved_sources', MOCK_SOURCES)
    synthesis = ctx.scratch.get('synthesis', {})
    
    # Mock fact checking
    verified_claims = []
    disputed_claims = []
    
    for src in sources:
        confidence = src['credibility_score']
        if confidence >= 0.80:
            verified_claims.append({
                'claim': src['key_finding'],
                'source': src['id'],
                'confidence': confidence,
                'verified': True,
            })
        elif confidence >= 0.70:
            verified_claims.append({
                'claim': src['key_finding'],
                'source': src['id'],
                'confidence': confidence,
                'note': 'Acceptable — single study, replicate needed',
                'verified': True,
            })
        else:
            disputed_claims.append({
                'claim': src['key_finding'],
                'source': src['id'],
                'confidence': confidence,
                'issue': 'Industry report, potential publication bias',
            })
    
    fact_check_score = len(verified_claims) / len(sources) if sources else 0
    
    result = {
        'agent': 'FactChecker',
        'verified_claims': len(verified_claims),
        'disputed_claims': len(disputed_claims),
        'fact_check_score': round(fact_check_score, 3),
        'verified': verified_claims,
        'disputed': disputed_claims,
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'fact_check': result},
        stop=False,
    )

fact_checker = AgentBuilder('FactChecker', 'fact-checker').with_step(_fact_checker_step).build()


# 2.5 QualityCritic — Evaluates synthesis quality and produces improvement suggestions

def _quality_critic_step(ctx: Context) -> AgentStep:
    """
    Evaluate the quality of the research synthesis.
    
    REAL LLM VERSION: uses GPT-4o as judge, scoring on: coherence,
    evidence coverage, balanced perspective, actionable conclusions.
    """
    synthesis  = ctx.scratch.get('synthesis', {})
    fact_check = ctx.scratch.get('fact_check', {})
    iteration  = ctx.scratch.get('critique_iteration', 0)
    
    # Simulate quality improving with each iteration
    base_quality = 0.62 + iteration * 0.08
    noise        = random.gauss(0, 0.02)
    quality_score = min(0.95, base_quality + noise)
    
    fact_coverage = fact_check.get('fact_check_score', 0.5)
    overall_quality = (quality_score * 0.6 + fact_coverage * 0.4)
    
    improvements = []
    if quality_score < 0.75:
        improvements.append('Add quantitative synthesis across study sample sizes')
    if fact_coverage < 0.80:
        improvements.append('Address disputed claims with additional sources')
    if iteration == 0:
        improvements.append('Add limitations section discussing external validity')
    
    result = {
        'agent': 'QualityCritic',
        'quality_score': round(quality_score, 4),
        'fact_coverage': round(fact_coverage, 4),
        'overall_quality': round(overall_quality, 4),
        'passed_threshold': overall_quality >= 0.75,
        'improvements_needed': improvements,
        'critique_iteration': iteration,
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'critique': result, 'critique_iteration': iteration + 1},
        stop=False,
    )

quality_critic = AgentBuilder('QualityCritic', 'critic').with_step(_quality_critic_step).build()
print('FactChecker + QualityCritic built.')

In [ ]:
# 2.6 ReportGenerator — Produces the final structured report

def _report_generator_step(ctx: Context) -> AgentStep:
    """
    Generate a polished research report.
    
    REAL LLM VERSION: GPT-4o receives full synthesis + fact-check + critique
    and generates a structured markdown report with executive summary,
    methodology section, findings, limitations, and recommendations.
    """
    synthesis   = ctx.scratch.get('synthesis', {})
    fact_check  = ctx.scratch.get('fact_check', {})
    critique    = ctx.scratch.get('critique', {})
    iterations  = ctx.scratch.get('critique_iteration', 1)
    memory      = ctx.memory  # accumulated research context
    
    report = {
        'title': f'Research Report: {RESEARCH_TOPIC}',
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'refinement_iterations': iterations,
        'executive_summary': synthesis.get('summary', ''),
        'key_findings': synthesis.get('key_findings', []),
        'evidence_base': {
            'total_sources': synthesis.get('sources_used', 0),
            'total_sample_size': synthesis.get('total_sample_size', 0),
            'avg_source_credibility': synthesis.get('avg_source_credibility', 0),
            'verified_claims': fact_check.get('verified_claims', 0),
            'disputed_claims': fact_check.get('disputed_claims', 0),
        },
        'quality_metrics': {
            'overall_quality': critique.get('overall_quality', 0),
            'fact_coverage': critique.get('fact_coverage', 0),
            'passed_review': critique.get('passed_threshold', False),
        },
        'recommendations': [
            'Adopt AI coding assistants for junior developer onboarding first',
            'Implement mandatory code review for LLM-generated code segments',
            'Track maintenance debt metrics 6 months post-AI adoption',
            'Invest in prompt engineering training for senior developers',
        ],
        'limitations': [
            'Most studies are <2 years post-LLM adoption; long-term effects unknown',
            'Sample populations skewed toward Western tech companies',
            'Self-report surveys may overstate productivity gains',
        ],
        'memory_context_used': len(memory) if memory else 0,
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(report))],
        state_updates={'final_report': report},
        stop=True,
    )

report_generator = AgentBuilder('ReportGenerator', 'report-writer').with_step(_report_generator_step).build()
print('ReportGenerator built.')

## Section 3 — DAG Construction with GraphRunner

We use `GraphRunner` to define the research workflow as a directed acyclic graph. The key advantage over a linear chain is that `summarize` and `fact_check` run **in parallel**, halving that stage's latency.

In [ ]:
# ── Parallel stage: summarize + fact_check ─────────────────────────────────────
parallel_analysis = MapReduceCoordinator(
    mappers=[summarizer, fact_checker],
    reducer=quality_critic,
)

def run_full_research_pipeline(
    topic: str,
    max_refinement_iterations: int = 5,
    quality_threshold: float = 0.75,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Run the complete research pipeline with iterative self-correction.
    
    Pipeline stages:
      1. TopicExtractor — decompose the question
      2. SourceSearcher — retrieve relevant literature
      3. [Parallel] SummarySynthesizer + FactChecker
      4. QualityCritic — evaluate and suggest improvements
      5. [Loop if quality < threshold] → refine
      6. ReportGenerator — produce final report
    """
    iteration_log = []
    memory_store  = {}  # simulates EpisodicMemory
    
    # ── Stage 1: Topic extraction
    ctx = Context(goal=f'Research: {topic}')
    ctx.scratch['research_topic'] = topic
    
    topic_result = topic_extractor.run(ctx)
    if verbose:
        print(f"[1] TopicExtractor: {len(ctx.scratch.get('topic_analysis', {}).get('sub_topics', []))} sub-topics identified")
    
    # ── Stage 2: Source retrieval
    source_result = source_searcher.run(ctx)
    n_sources = len(ctx.scratch.get('retrieved_sources', []))
    if verbose:
        print(f"[2] SourceSearcher: {n_sources} sources retrieved")
    
    # ── Stage 3 + 4: Parallel analysis + quality critique (iterative)
    ctx.scratch['critique_iteration'] = 0
    passed_review = False
    
    for i in range(max_refinement_iterations):
        # Parallel summarize + fact_check, then critique
        ctx.scratch['shards'] = ['full_corpus']
        analysis_out = parallel_analysis.run(goal=ctx.goal, inputs=ctx.scratch)
        
        critique = ctx.scratch.get('critique', {})
        quality  = critique.get('overall_quality', 0)
        passed   = critique.get('passed_threshold', False)
        
        iteration_log.append({
            'iteration': i + 1,
            'quality_score': quality,
            'passed_threshold': passed,
            'improvements': critique.get('improvements_needed', []),
        })
        
        if verbose:
            print(f"[3/{i+1}] QualityCritic: quality={quality:.3f}  passed={'YES' if passed else 'NO'}")
        
        # Accumulate critique in memory for context in next iteration
        memory_store[f'critique_iter_{i}'] = critique
        ctx.memory = memory_store
        
        if passed:
            passed_review = True
            break
    
    # ── Stage 5: Generate final report
    report_result = report_generator.run(ctx)
    final_report  = ctx.scratch.get('final_report', {})
    
    if verbose:
        print(f"[4] ReportGenerator: report generated ({len(str(final_report))} chars)")
        print(f"\nPassed quality review: {'YES' if passed_review else 'NO (max iterations reached)'}")
    
    return {
        'report': final_report,
        'iteration_log': iteration_log,
        'passed_review': passed_review,
    }


# Run the pipeline
print(f'Running research pipeline on: "{RESEARCH_TOPIC}"\n')
pipeline_output = run_full_research_pipeline(RESEARCH_TOPIC)

## Section 4 — Iterative Refinement Analysis

We visualise how quality improves across refinement iterations.

In [ ]:
print("\nRefinement Iteration Log:")
print(f"{'Iter':>4}  {'Quality':>8}  {'Passed':>7}  Improvements")
print('-' * 65)

for entry in pipeline_output['iteration_log']:
    improvements_str = '; '.join(entry['improvements'][:2]) if entry['improvements'] else 'None'
    print(f"{entry['iteration']:>4}  {entry['quality_score']:>8.4f}  {'YES':>7 if entry['passed_threshold'] else 'NO':>7}  {improvements_str[:45]}")

print(f"\nFinal outcome: {'PASSED' if pipeline_output['passed_review'] else 'MAX ITERATIONS'}")

## Section 5 — Ensemble Agreement Score

Run the full pipeline N times to measure variance in quality scores. High variance means the pipeline quality depends heavily on the randomness of the mock LLM (in production: temperature of the real LLM). Low variance = stable, reliable pipeline.

In [ ]:
def compute_research_confidence(topic: str, n_runs: int = 5) -> Dict[str, Any]:
    """Run the pipeline n_runs times and compute quality statistics."""
    quality_scores = []
    passed_count   = 0
    
    for run_idx in range(n_runs):
        result = run_full_research_pipeline(topic, verbose=False)
        # Get final quality from last iteration
        if result['iteration_log']:
            final_q = result['iteration_log'][-1]['quality_score']
            quality_scores.append(final_q)
        if result['passed_review']:
            passed_count += 1
    
    mean_q = statistics.mean(quality_scores) if quality_scores else 0
    std_q  = statistics.stdev(quality_scores) if len(quality_scores) > 1 else 0
    
    return {
        'n_runs': n_runs,
        'quality_scores': quality_scores,
        'mean_quality': round(mean_q, 4),
        'std_quality': round(std_q, 4),
        'pass_rate': passed_count / n_runs,
        'agreement_score': round(1.0 - std_q / max(mean_q, 0.001), 4),
        'research_confidence': round(mean_q * (passed_count / n_runs), 4),
    }


confidence_result = compute_research_confidence(RESEARCH_TOPIC, n_runs=5)

print(f"Research Pipeline Confidence Analysis")
print(f"Topic: {RESEARCH_TOPIC}")
print(f"\nRuns: {confidence_result['n_runs']}")
print(f"Quality scores: {[f'{q:.3f}' for q in confidence_result['quality_scores']]}")
print(f"Mean quality    : {confidence_result['mean_quality']:.4f}")
print(f"Std quality     : {confidence_result['std_quality']:.4f}")
print(f"Pass rate       : {confidence_result['pass_rate']:.0%}")
print(f"Agreement score : {confidence_result['agreement_score']:.4f}")
print(f"Research confidence: {confidence_result['research_confidence']:.4f}")

## Section 6 — Final Report Output

In [ ]:
report = pipeline_output['report']

print(f"{'='*70}")
print(f"RESEARCH REPORT")
print(f"{'='*70}")
print(f"Title: {report.get('title', '')}")
print(f"Generated: {report.get('generated_at', '')}")
print(f"Refinement iterations: {report.get('refinement_iterations', 0)}")
print(f"\nEXECUTIVE SUMMARY")
print(f"{report.get('executive_summary', '')}")
print(f"\nKEY FINDINGS ({len(report.get('key_findings', []))})")
for i, finding in enumerate(report.get('key_findings', []), 1):
    print(f"  {i}. {finding}")
print(f"\nEVIDENCE BASE")
eb = report.get('evidence_base', {})
print(f"  Sources: {eb.get('total_sources', 0)} | N={eb.get('total_sample_size', 0):,} | Avg credibility={eb.get('avg_source_credibility', 0):.2f}")
print(f"  Verified claims: {eb.get('verified_claims', 0)} | Disputed: {eb.get('disputed_claims', 0)}")
print(f"\nQUALITY METRICS")
qm = report.get('quality_metrics', {})
print(f"  Overall quality: {qm.get('overall_quality', 0):.4f} | Fact coverage: {qm.get('fact_coverage', 0):.4f}")
print(f"  Passed review: {'YES' if qm.get('passed_review') else 'NO'}")
print(f"\nRECOMMENDATIONS")
for rec in report.get('recommendations', []):
    print(f"  • {rec}")
print(f"\nLIMITATIONS")
for lim in report.get('limitations', []):
    print(f"  • {lim}")

## Section 7 — Summary

### Patterns demonstrated

| Pattern | Class | Purpose |
|---------|-------|----------|
| Parallel execution | `MapReduceCoordinator` | Summarize + FactCheck concurrently |
| Sequential stages | `AssemblyCoordinator` / manual | Linear pipeline progression |
| Iterative refinement | Custom FSM loop | Draft → critique → revise |
| Episodic memory | `ctx.memory` dict | Accumulate critique history |
| Ensemble confidence | Multiple pipeline runs | Stability measurement |

### Production readiness checklist

- [ ] Replace `FunctionAdapter` steps with real LLM calls (`EnvOpenAIAdapter`)
- [ ] Add `EpisodicMemory` with SQLite backend for persistent session state
- [ ] Wire `SourceSearcher` to Semantic Scholar API + vector DB for real retrieval
- [ ] Add `TwoPersonRuleCoordinator` for peer review on high-stakes reports
- [ ] Export quality metrics to Prometheus dashboard